In [1]:
import os
from typing import Dict, List, Optional
import pandas as pd


def combine_paired_csv_folders_ignore_first_token(
    folder_a: str,
    folder_b: str,
    output_parent_dir: Optional[str] = None,
    output_folder_name: str = "Bio Rep 1_3",
    file_encoding: str = "utf-8",
    on_duplicates_in_folder: str = "error",  # "error" | "first"
) -> Dict[str, str]:
    """
    For each paired key, write ONE output CSV where the content is:
        [all rows from folder_a file] appended with [all rows from folder_b file]
    i.e., a literal vertical concatenation.

    Pairing rule:
      - Ignore the first '_' token of the filename.
      - key = everything after the first token, plus extension.
        "1_48hr_Area.csv" and "3_48hr_Area.csv" -> key "48hr_Area.csv"

    Do not skip anything:
      - If a key exists in only one folder -> still write an output CSV for that key.
        (It will contain only the rows from the folder that has it.)
      - If duplicates exist within a folder for the same key:
          - "error": raise and stop (safest)
          - "first": pick the first (sorted) and continue
    """
    folder_a = os.path.abspath(folder_a)
    folder_b = os.path.abspath(folder_b)

    if not os.path.isdir(folder_a):
        raise FileNotFoundError(f"folder_a not found: {folder_a}")
    if not os.path.isdir(folder_b):
        raise FileNotFoundError(f"folder_b not found: {folder_b}")

    if output_parent_dir is None:
        output_parent_dir = os.path.dirname(folder_a)
    output_parent_dir = os.path.abspath(output_parent_dir)

    out_dir = os.path.join(output_parent_dir, output_folder_name)
    os.makedirs(out_dir, exist_ok=True)

    def list_csvs(folder: str) -> List[str]:
        return sorted(
            [
                os.path.join(folder, f)
                for f in os.listdir(folder)
                if f.lower().endswith(".csv") and not f.startswith("._")
            ]
        )

    def pairing_key_from_filename(path: str) -> str:
        base = os.path.basename(path)
        stem, ext = os.path.splitext(base)

        if "_" not in stem:
            return base

        tokens = [t for t in stem.split("_") if t != ""]
        if len(tokens) < 2:
            return base

        return "_".join(tokens[1:]) + ext

    def index_folder(folder: str) -> Dict[str, List[str]]:
        idx: Dict[str, List[str]] = {}
        for p in list_csvs(folder):
            idx.setdefault(pairing_key_from_filename(p), []).append(p)
        return idx

    def pick_one(paths: List[str], *, folder: str, key: str) -> str:
        if len(paths) == 1:
            return paths[0]
        paths_sorted = sorted(paths)
        if on_duplicates_in_folder == "first":
            return paths_sorted[0]
        raise ValueError(
            f"Duplicate pairing key within folder.\n"
            f"Folder: {folder}\nKey: {key}\nMatches:\n  - " + "\n  - ".join(paths_sorted)
        )

    idx_a = index_folder(folder_a)
    idx_b = index_folder(folder_b)

    # IMPORTANT: do not skip anything -> union of keys
    all_keys = sorted(set(idx_a.keys()) | set(idx_b.keys()))
    outputs: Dict[str, str] = {}

    for key in all_keys:
        parts: List[pd.DataFrame] = []

        if key in idx_a:
            path_a = pick_one(idx_a[key], folder=folder_a, key=key)
            df_a = pd.read_csv(path_a, encoding=file_encoding)
            parts.append(df_a)

        if key in idx_b:
            path_b = pick_one(idx_b[key], folder=folder_b, key=key)
            df_b = pd.read_csv(path_b, encoding=file_encoding)
            parts.append(df_b)

        if not parts:
            # Should not happen because key came from union
            continue

        out_df = pd.concat(parts, axis=0, ignore_index=True, sort=False)

        out_path = os.path.join(out_dir, key)  # keep the new generated csv file name == key
        out_df.to_csv(out_path, index=False)
        outputs[key] = out_path

    print(f"Folder A keys: {len(idx_a)} | Folder B keys: {len(idx_b)} | Total output keys: {len(all_keys)}")
    print(f"Wrote {len(outputs)} combined CSV(s) to:\n  {out_dir}")
    return outputs


# Call
combine_paired_csv_folders_ignore_first_token(
    folder_a='/Users/allisonpickle/Desktop/Bio Rep 1_3/260415',
    folder_b='/Users/allisonpickle/Desktop/Bio Rep 1_3/260601',
    output_parent_dir="/Users/allisonpickle/Desktop",
    output_folder_name="Bio Rep 1_3",
    on_duplicates_in_folder="error",  # set "first" only if you truly want arbitrary selection
)

Folder A keys: 8 | Folder B keys: 8 | Total output keys: 9
Wrote 9 combined CSV(s) to:
  /Users/allisonpickle/Desktop/Bio Rep 1_3


{'Area (µm²).csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Area (µm²).csv',
 'Aspect Ratio.csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Aspect Ratio.csv',
 'Base Name.csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Base Name.csv',
 'Circularity.csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Circularity.csv',
 'Convex Hull Area (µm²).csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Convex Hull Area (µm²).csv',
 'Convex Hull Perimeter (µm).csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Convex Hull Perimeter (µm).csv',
 'Microglia Morphology_Sheet1.csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Microglia Morphology_Sheet1.csv',
 'Perimeter (µm).csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Perimeter (µm).csv',
 'Roughness.csv': '/Users/allisonpickle/Desktop/Bio Rep 1_3/Roughness.csv'}